# Analyse exploratoire et modélisation – Boston Housing

**Légende** : 🔧 = cellule corrigée par rapport à votre version · ➕ = cellule ajoutée pour compléter l'analyse.

**Dictionnaire des variables**

| Variable | Signification |
|---|---|
| CRIM | Taux de criminalité par habitant |
| ZN | % de terrains résidentiels de grande taille (> 25 000 sq.ft) |
| INDUS | % de surfaces commerciales non-détail (industrie) |
| CHAS | 1 si le quartier borde la rivière Charles, 0 sinon |
| NOX | Concentration en oxydes d'azote (pollution) |
| RM | Nombre moyen de pièces par logement |
| AGE | % de logements construits **avant 1940** (ce n'est pas un âge en années) |
| DIS | Distance pondérée aux centres d'emploi |
| RAD | Indice d'accessibilité aux autoroutes |
| TAX | Taux de taxe foncière (pour 10 000 $) |
| PTRATIO | Ratio élèves / enseignants |
| B | Variable construite à partir de la proportion de population noire (⚠️ éthiquement problématique, exclue de la modélisation) |
| LSTAT | % de population à faible statut socio-économique |
| **MEDV** | **Valeur médiane des logements (en milliers de $) : variable cible** |

## 1. Imports et chargement des données

In [ ]:
## 🔧 Bibliothèques (seaborn n'est importé qu'une fois)
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
## Importer la base de données (le fichier HousingData.csv doit être dans le même dossier que le notebook)
HousingData = pd.read_csv('HousingData.csv', sep=';')

## 2. Première exploration

In [ ]:
## Dimension de la base de données
HousingData.shape

In [ ]:
## Type de données
HousingData.dtypes

In [ ]:
## Aperçu des premières lignes
HousingData.head(8)

In [ ]:
## Noms des colonnes
HousingData.columns

In [ ]:
## ➕ Informations générales (types + valeurs non nulles)
HousingData.info()

In [ ]:
## Statistiques descriptives
## À regarder : écart moyenne/médiane (asymétrie), min/max (extrêmes), count (valeurs manquantes)
HousingData.describe().T

## 3. Qualité des données : valeurs manquantes et doublons

In [ ]:
## 🔧 Valeurs nulles : nombre ET pourcentage par colonne
manquantes = pd.DataFrame({
    'nb_manquantes': HousingData.isnull().sum(),
    'pourcentage': (HousingData.isnull().mean() * 100).round(2)
})
manquantes[manquantes['nb_manquantes'] > 0]

In [ ]:
## ➕ Visualisation des valeurs manquantes
plt.figure(figsize=(10, 4))
sns.heatmap(HousingData.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Valeurs manquantes (jaune = manquant)')
plt.show()

print('Lignes avec au moins une valeur manquante :', HousingData.isnull().any(axis=1).sum())

In [ ]:
## Doublons : méthode duplicated()
HousingData.duplicated().sum()

## 4. Détection des valeurs aberrantes
On détecte **avant** de modifier les données. Deux méthodes : IQR (robuste, recommandée pour des variables asymétriques) et z-score (suppose une distribution proche de la normale).

In [ ]:
## Boxplot de CRIM
plt.figure(figsize=(8, 3))
sns.boxplot(x=HousingData['CRIM'])
plt.title('CRIM')
plt.show()

In [ ]:
## ➕ Boxplots de toutes les variables numériques
cols_num = HousingData.select_dtypes(include='number').columns.drop('CHAS')
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, col in zip(axes.ravel(), cols_num):
    sns.boxplot(y=HousingData[col], ax=ax)
    ax.set_title(col)
for ax in axes.ravel()[len(cols_num):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
## 🔧 Fonctions de détection (réutilisables pour n'importe quelle colonne)
def bornes_iqr(serie, k=1.5):
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    # CORRECTION : la borne basse part de Q1 (et non de Q3 comme dans la version initiale)
    return q1 - k * iqr, q3 + k * iqr

def bornes_zscore(serie, seuil=3):
    s = serie.dropna()
    return s.mean() - seuil * s.std(), s.mean() + seuil * s.std()

def aberrantes(df, col, methode='iqr'):
    inf, sup = bornes_iqr(df[col]) if methode == 'iqr' else bornes_zscore(df[col])
    return df.loc[(df[col] < inf) | (df[col] > sup)]

In [ ]:
## 🔧 Détection pour LSTAT (IQR) – borne basse corrigée
inf, sup = bornes_iqr(HousingData['LSTAT'])
print('Borne inférieure :', round(inf, 2))
print('Borne supérieure :', round(sup, 2))
display(aberrantes(HousingData, 'LSTAT', 'iqr'))

In [ ]:
## 🔧 Détection par z-score pour AGE, CRIM et TAX (une seule cellule, plus de doublons)
for col in ['AGE', 'CRIM', 'TAX']:
    inf, sup = bornes_zscore(HousingData[col])
    ab = aberrantes(HousingData, col, 'zscore')
    print(f'{col} : bornes [{inf:.2f} ; {sup:.2f}] -> {len(ab)} valeurs aberrantes')

In [ ]:
## ➕ Tableau récapitulatif des valeurs aberrantes (IQR et z-score) pour toutes les variables
lignes = []
for col in cols_num:
    lignes.append({
        'variable': col,
        'aberrantes_IQR': len(aberrantes(HousingData, col, 'iqr')),
        'aberrantes_zscore': len(aberrantes(HousingData, col, 'zscore')),
        'asymetrie (skew)': round(HousingData[col].skew(), 2)
    })
resume_ab = pd.DataFrame(lignes).sort_values('aberrantes_IQR', ascending=False)
resume_ab

**Décision sur les valeurs aberrantes** : on ne les supprime pas. Ce sont des quartiers réels (par ex. forte criminalité), donc de l'information utile. On réduit plutôt leur influence par une transformation logarithmique de `CRIM` (section 6) et par des modèles robustes (section 11).

## 5. Traitement des valeurs manquantes
🔧 On impute par la **médiane** (robuste aux valeurs extrêmes) et pour **toutes** les colonnes concernées : `CRIM`, `ZN`, `INDUS`, `CHAS`, `AGE`, `LSTAT`. La version initiale ne traitait que `AGE`, `CRIM` et `TAX` (cette dernière n'avait pourtant aucune valeur manquante).

In [ ]:
## Avant : on garde une trace des moyennes/médianes pour comparer
avant = HousingData[['CRIM', 'AGE', 'LSTAT']].describe().loc[['mean', '50%', 'std']]

colonnes_a_imputer = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'AGE', 'LSTAT']
for col in colonnes_a_imputer:
    HousingData[col] = HousingData[col].fillna(HousingData[col].median())

# CHAS est une variable binaire : on la remet en entier
HousingData['CHAS'] = HousingData['CHAS'].astype(int)

apres = HousingData[['CRIM', 'AGE', 'LSTAT']].describe().loc[['mean', '50%', 'std']]
print('AVANT imputation'); display(avant)
print('APRÈS imputation'); display(apres)

assert HousingData.isnull().sum().sum() == 0, 'Il reste des valeurs manquantes'
print('✔ Plus aucune valeur manquante')

## 6. Création de nouvelles variables

In [ ]:
## 🔧 classe_age avec pd.cut (plus rapide que la boucle, étiquettes corrigées)
## Rappel : AGE = % de logements construits avant 1940
HousingData['classe_age'] = pd.cut(
    HousingData['AGE'],
    bins=[0, 30, 50, 70, 90, 100],
    labels=['0-30 %', '30-50 %', '50-70 %', '70-90 %', '90-100 %'],
    include_lowest=True
)
HousingData['classe_age'].value_counts().sort_index()

In [ ]:
## ➕ log_CRIM : réduit l'asymétrie extrême de CRIM
HousingData['log_CRIM'] = np.log1p(HousingData['CRIM'])
print('Asymétrie CRIM     :', round(HousingData['CRIM'].skew(), 2))
print('Asymétrie log_CRIM :', round(HousingData['log_CRIM'].skew(), 2))

In [ ]:
## ➕ Indicateurs complémentaires issus de l'exploration
## - MEDV_plafonne : MEDV est plafonné à 50 (observations « censurées »)
## - groupe_RAD24 : groupe de quartiers très connectés aux autoroutes (RAD = 24, TAX élevé)
HousingData['MEDV_plafonne'] = (HousingData['MEDV'] >= 50).astype(int)
HousingData['groupe_RAD24'] = (HousingData['RAD'] == 24).astype(int)

print('Quartiers avec MEDV plafonné à 50 :', HousingData['MEDV_plafonne'].sum())
print('Quartiers du groupe RAD = 24      :', HousingData['groupe_RAD24'].sum())

## 7. Visualisation des distributions

In [ ]:
## Distribution de AGE
plt.hist(HousingData['AGE'], bins=15, edgecolor='k')
plt.xlabel('AGE (% de logements construits avant 1940)')
plt.ylabel('Fréquence')
plt.title('Distribution de AGE')
plt.show()

In [ ]:
## Distribution de LSTAT
plt.hist(HousingData['LSTAT'], bins=15, edgecolor='black')
plt.xlabel('LSTAT (% population défavorisée)')
plt.ylabel('Fréquence')
plt.title('Distribution de LSTAT')
plt.show()

In [ ]:
## LSTAT selon CHAS
plt.figure(figsize=(9, 5))
sns.boxplot(data=HousingData, x='CHAS', y='LSTAT')
plt.title('LSTAT selon CHAS')
plt.xlabel('CHAS (0 = hors rivière, 1 = bord de rivière)')
plt.ylabel('LSTAT')
plt.show()

In [ ]:
## Histogrammes de toutes les variables numériques d'origine
colonnes_numeriques = HousingData.select_dtypes(include='number').drop(
    columns=['log_CRIM', 'MEDV_plafonne', 'groupe_RAD24'])
colonnes_numeriques.hist(bins=20, figsize=(16, 12), color='red', edgecolor='black', grid=False)
plt.suptitle('Analyse des données HousingData')
plt.tight_layout()
plt.show()

In [ ]:
## 🔧 Répartition de CHAS (variable binaire) – le code traçait CRIM par erreur
plt.figure(figsize=(7, 5))
sns.countplot(data=HousingData, x='CHAS')
plt.title('Répartition de la variable CHAS')
plt.xlabel('CHAS')
plt.ylabel("Nombre d'observations")
plt.show()

In [ ]:
## Comparaison des 4 variables AGE, TAX, LSTAT, CRIM
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, ['AGE', 'TAX', 'LSTAT', 'CRIM']):
    ax.hist(HousingData[col], bins=15, edgecolor='black')
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
## Proportion de la variable CHAS
chas_counts = HousingData['CHAS'].value_counts().sort_index()
plt.pie(chas_counts.values, labels=['Hors rivière (0)', 'Bord de rivière (1)'],
        autopct='%1.1f%%', startangle=140)
plt.axis('equal')
plt.title('Répartition de la variable CHAS')
plt.show()

In [ ]:
## 🔧 Répartition en 4 intervalles de chaque variable (version anneau)
## Correction : CHAS exclue (binaire), figures fermées après affichage.
## Remarque : pour des variables continues, les histogrammes ci-dessus restent plus lisibles.
for colonne in colonnes_numeriques.columns.drop('CHAS'):
    intervalles = pd.cut(HousingData[colonne], bins=4)
    repartition = intervalles.value_counts().sort_index()
    pourcentages = repartition / repartition.sum() * 100
    legendes = [f'{i}: {e} obs. ({p:.1f}%)' for i, e, p in zip(repartition.index, repartition.values, pourcentages.values)]

    fig, ax = plt.subplots(figsize=(12, 6))
    wedges, texts, autotexts = ax.pie(repartition.values, autopct='%1.1f%%', startangle=90,
                                      pctdistance=0.75, textprops={'fontsize': 11},
                                      wedgeprops={'width': 0.45, 'edgecolor': 'white'})
    for t in autotexts:
        t.set_fontweight('bold')
    ax.legend(wedges, legendes, title='Répartition des données', loc='center left', bbox_to_anchor=(1, 0.5))
    ax.set_title(f'Répartition de la variable : {colonne}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
## Répartition de RAD (accessibilité aux autoroutes) en pourcentage
rad_pct = HousingData['RAD'].value_counts(normalize=True).sort_index() * 100

plt.figure(figsize=(10, 6))
bars = plt.bar(rad_pct.index.astype(str), rad_pct.values)
plt.xlabel('Indice RAD')
plt.ylabel("Pourcentage d'observations (%)")
plt.title('Répartition de la variable RAD')
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 0.5, f'{h:.1f}%', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 8. Corrélations

In [ ]:
## 🔧 Matrice de corrélation (une seule version, triangle inférieur pour la lisibilité)
variables_corr = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD',
                  'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
correlation = HousingData[variables_corr].corr()

masque = np.triu(np.ones_like(correlation, dtype=bool))
plt.figure(figsize=(12, 8))
sns.heatmap(correlation, mask=masque, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Matrice de corrélation – HousingData')
plt.show()

In [ ]:
## ➕ Corrélation de chaque variable avec la cible MEDV
corr_medv = correlation['MEDV'].drop('MEDV').sort_values()
plt.figure(figsize=(8, 5))
corr_medv.plot.barh(color=['firebrick' if v < 0 else 'steelblue' for v in corr_medv])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Corrélation avec MEDV')
plt.xlabel('Coefficient de Pearson')
plt.show()
corr_medv.sort_values(key=abs, ascending=False).round(2)

In [ ]:
## ➕ Paires de variables explicatives fortement corrélées (|r| > 0,7) -> risque de multicolinéarité
paires = (correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
          .stack().rename('r').reset_index())
paires.columns = ['var1', 'var2', 'r']
paires = paires[(paires['var1'] != 'MEDV') & (paires['var2'] != 'MEDV') & (paires['r'].abs() > 0.7)]
paires.sort_values('r', key=abs, ascending=False).round(2)

In [ ]:
## ➕ Corrélation de Spearman (robuste aux valeurs extrêmes et aux relations non linéaires)
spearman_medv = HousingData[variables_corr].corr(method='spearman')['MEDV'].drop('MEDV')
comparaison = pd.DataFrame({'Pearson': corr_medv, 'Spearman': spearman_medv}).round(2)
comparaison.sort_values('Spearman', key=abs, ascending=False)

## 9. Relations avec la variable cible MEDV

In [ ]:
## ➕ Distribution de la cible MEDV
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(HousingData['MEDV'], bins=25, kde=True, ax=axes[0])
axes[0].set_title('Distribution de MEDV (note le pic à 50)')
sns.boxplot(x=HousingData['MEDV'], ax=axes[1])
axes[1].set_title('Boxplot de MEDV')
plt.tight_layout()
plt.show()

In [ ]:
## ➕ Relation de MEDV avec les variables les plus corrélées
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, v in zip(axes.ravel(), ['LSTAT', 'RM', 'PTRATIO', 'INDUS', 'NOX', 'log_CRIM']):
    sns.regplot(data=HousingData, x=v, y='MEDV', ax=ax,
                scatter_kws={'s': 14, 'alpha': 0.5}, line_kws={'color': 'red'})
    ax.set_title(f'MEDV vs {v}')
plt.tight_layout()
plt.show()

In [ ]:
## ➕ MEDV selon CHAS et selon classe_age
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=HousingData, x='CHAS', y='MEDV', ax=axes[0])
axes[0].set_title('MEDV selon CHAS')
sns.boxplot(data=HousingData, x='classe_age', y='MEDV', ax=axes[1])
axes[1].set_title("MEDV selon la part de logements d'avant 1940")
plt.tight_layout()
plt.show()

HousingData.groupby('classe_age', observed=True)['MEDV'].agg(['count', 'mean', 'median']).round(2)

In [ ]:
## ➕ Le groupe RAD = 24 est-il différent des autres quartiers ?
profil = HousingData.groupby('groupe_RAD24')[['MEDV', 'CRIM', 'TAX', 'PTRATIO', 'INDUS', 'NOX', 'LSTAT', 'RM']].median().round(2)
profil.index = ['Autres quartiers', 'Groupe RAD = 24']
profil

In [ ]:
## ➕ Tests statistiques
## 1) Les quartiers au bord de la rivière ont-ils un MEDV différent ? (Mann-Whitney : sans hypothèse de normalité)
g0 = HousingData.loc[HousingData['CHAS'] == 0, 'MEDV']
g1 = HousingData.loc[HousingData['CHAS'] == 1, 'MEDV']
u, p = stats.mannwhitneyu(g0, g1)
print(f'CHAS : médiane MEDV hors rivière = {g0.median():.1f}, bord de rivière = {g1.median():.1f}, p-value = {p:.4f}')

## 2) Même question pour le groupe RAD = 24
a = HousingData.loc[HousingData['groupe_RAD24'] == 0, 'MEDV']
b = HousingData.loc[HousingData['groupe_RAD24'] == 1, 'MEDV']
u, p = stats.mannwhitneyu(a, b)
print(f'RAD=24 : médiane MEDV autres = {a.median():.1f}, groupe RAD=24 = {b.median():.1f}, p-value = {p:.4g}')

## 3) Corrélation de Spearman LSTAT / MEDV et RM / MEDV
for v in ['LSTAT', 'RM']:
    rho, p = stats.spearmanr(HousingData[v], HousingData['MEDV'])
    print(f'{v} vs MEDV : rho = {rho:.2f}, p-value = {p:.2g}')
print('\nRègle : p-value < 0,05 -> différence/corrélation statistiquement significative.')

## 10. Multicolinéarité (VIF)
Le VIF mesure à quel point une variable est expliquée par les autres. Règle usuelle : **VIF > 5** (ou 10) = problème.
On exclut `B` (éthique) et on utilise `log_CRIM` à la place de `CRIM`. Cette analyse sert à identifier les variables redondantes, sans construire de modèle.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

variables_exclues = ['MEDV', 'B', 'CRIM', 'classe_age', 'MEDV_plafonne', 'groupe_RAD24']
X_all = HousingData.drop(columns=variables_exclues)

def calcul_vif(X):
    Xs = (X - X.mean()) / X.std()
    Xs.insert(0, 'const', 1.0)
    vif = pd.Series([variance_inflation_factor(Xs.values, i) for i in range(1, Xs.shape[1])],
                    index=X.columns)
    return vif.sort_values(ascending=False).round(2)

print('VIF – toutes les variables'); display(calcul_vif(X_all))

## On retire la variable la plus problématique (RAD/TAX sont corrélées à 0,91) et on recalcule
a_exclure = ['TAX']          # <- à modifier selon vos choix (ex. ['RAD'])
X_reduit = X_all.drop(columns=a_exclure)
print(f'VIF – après exclusion de {a_exclure}'); display(calcul_vif(X_reduit))

## 11. Synthèse et conclusion

> ✍️ **À compléter avec vos résultats** après avoir exécuté le notebook en entier.

**Constats établis par l'analyse**
- Jeu de données de 506 quartiers, sans doublon, avec environ 4 % de valeurs manquantes sur 6 variables (imputées par la médiane).
- Facteurs les plus liés au prix (`MEDV`) : **LSTAT (r ≈ −0,74)** et **RM (r ≈ +0,70)**, puis PTRATIO (−0,51), INDUS (−0,48), TAX (−0,47), NOX (−0,43).
- `CHAS` (bord de rivière) : effet faible, ≈ 7 % des quartiers. Résultat du test de Mann-Whitney : ……
- Colinéarité forte entre `RAD` et `TAX` (r ≈ 0,91) ; groupe distinct de quartiers (RAD = 24, TAX ≈ 666, PTRATIO ≈ 20,2). Résultat du profil et du test : ……
- Variables les plus asymétriques / avec le plus de valeurs aberrantes (tableau section 4) : ……
- `MEDV` est plafonné à 50 ; `CRIM` est très asymétrique (réduit par `log_CRIM`).

**Recommandations**
1. Retenir `LSTAT` et `RM` comme indicateurs principaux du niveau de prix d'un quartier.
2. Ne pas utiliser simultanément `RAD` et `TAX` (redondantes).
3. Ne pas utiliser la variable `B` pour des décisions réelles (problème éthique).
4. Interpréter avec prudence les valeurs de `MEDV` proches de 50 (plafonnées).

**Limites** : données historiques (années 1970) et agrégées par quartier ; une corrélation ne prouve pas une causalité.